# Delta Lake Fundamentals

Delta Lake is the default data storage format in Databricks. It adds ACID transactions, schema enforcement and time travel to your tables. You create Delta tables, change them with DML and MERGE, read and restore old versions with Time Travel, read row-level changes with Change Data Feed, and compare Managed and External Tables.

> Every table you create in Databricks is a Delta table by default. Delta = Parquet files + transaction log (`_delta_log`).

## Learning Objectives

After completing this module you will be able to:

- Explain what Delta Lake is: Parquet files + the `_delta_log` transaction log
- Create a Delta table with `saveAsTable()` and explain schema enforcement and schema evolution
- Add a `CHECK` constraint and see it reject bad rows
- Perform `INSERT`, `UPDATE`, `DELETE` and `MERGE INTO` (upsert)
- Read table history with `DESCRIBE HISTORY` and query old versions with `VERSION AS OF` / `TIMESTAMP AS OF`
- Recover a table with `RESTORE` and explain how `VACUUM` and the retention limit time travel
- Read row-level changes with Change Data Feed and `table_changes()`
- Explain managed vs external tables: who owns the files, what `DROP TABLE` does, and how `SET MANAGED` converts an external table

## Setup

In [0]:
%run ../../setup/00_setup

### Configuration

In [0]:
from pyspark.sql import functions as F
spark.sql(f"USE CATALOG {CATALOG}")
spark.sql(f"USE SCHEMA {BRONZE_SCHEMA}")
print(f"Current schema: {CATALOG}.{BRONZE_SCHEMA}")

## Delta Lake Core Features

Delta Lake is a transaction layer over Parquet files. Delta records every write as a JSON commit in the `_delta_log/` folder. This gives you ACID transactions: a write is either fully visible or not visible at all, and readers never see a half-written table. On top of the log, Delta adds schema enforcement, schema evolution and constraints.

<img src="../../../assets/images/nb03-delta-core.png" alt="Delta Lake is Parquet files plus a transaction log: the writer writes Parquet files and commits one JSON file in _delta_log, the reader reads the log and only the current files; a file removed in the log stays until VACUUM; ACID transactions, schema enforcement and evolution, constraints, time travel, MERGE and batch plus streaming are built on the log" width="800">

### Creating the First Delta Table

A pipeline often starts by loading a raw file into a table. Load the customers CSV and save it as a managed table. `saveAsTable` without a path creates a managed Unity Catalog table, and Delta is the default format.

In [0]:
# Read the customers CSV from the Unity Catalog volume
customers_df = (spark.read
    .option("header", "true")
    .option("inferSchema", "true")
    .csv(f"{DATASET_PATH}/customers/customers.csv")
)

In [0]:
# Create the managed Delta table; DROP first, so a rerun does not keep the CHECK constraint below
CD = f"{CATALOG}.{BRONZE_SCHEMA}.customers_delta"
spark.sql(f"DROP TABLE IF EXISTS {CD}")
customers_df.write.format("delta").mode("overwrite").saveAsTable(CD)

In [0]:
display(spark.table(CD).limit(5))

### Schema Enforcement

<img src="../../../assets/images/nb03-schema-enforcement-v2.png" alt="Schema enforcement by default: an unknown column or incompatible type raises AnalysisException, missing columns are stored as NULL; schema evolution with mergeSchema adds new columns; type widening for widening changes, overwriteSchema for the rest" width="800">

Schema enforcement protects a table from a source that suddenly sends other columns. Delta compares the schema of incoming data with the table schema. It rejects a write with an unknown column or an incompatible type. Missing columns are fine: Delta stores them as `NULL`. The bad data below differs from a valid row only by one extra column, `loyalty_points`.

In [0]:
spark.table(CD).printSchema()

In [0]:
# One valid row with a new id, plus a column the table does not have
valid_data = customers_df.limit(1).withColumn("customer_id", F.lit("CUST999999"))
bad_data = valid_data.withColumn("loyalty_points", F.lit(120))
bad_data.printSchema()

In [0]:
# Schema enforcement rejects the unknown column (unless schema evolution is enabled)
from pyspark.errors import PySparkException

try:
    bad_data.write.mode("append").saveAsTable(CD)
except PySparkException as e:
    msg = str(e)
    print(msg[max(msg.find("["), 0):].splitlines()[0])  # skip the Java exception name serverless may add
else:
    print("Not rejected: the write with an unknown column was accepted")

In [0]:
# Without the extra column the schema matches and the append succeeds
valid_data.write.mode("append").saveAsTable(CD)

### Schema Evolution

Sometimes a source gets a new column that you want to keep. To add a column on purpose, write with `mergeSchema = true`. Delta adds the new column to the table, and old rows get `NULL` in it.

In [0]:
# Two customers with an extra column, customer_tier
cols = ["customer_id", "first_name", "last_name", "email", "phone", "city", "state",
        "country", "registration_date", "customer_segment", "customer_tier"]
extended_customers = spark.createDataFrame([
    ("CUST010001", "New", "Customer", "new@example.com", "+48 111 222 333",
     "Warsaw", "MZ", "Poland", "2023-12-01", "Basic", "Premium"),
    ("CUST010002", "Another", "Customer", "another@example.com", "+48 444 555 666",
     "Krakow", "MP", "Poland", "2023-12-02", "Premium", "Standard"),
], cols).withColumn("registration_date", F.to_date("registration_date"))
display(extended_customers.select("customer_id", "customer_segment", "customer_tier"))

In [0]:
# mergeSchema adds customer_tier to the table schema
extended_customers.write.mode("append").option("mergeSchema", "true").saveAsTable(CD)

In [0]:
# New schema: customer_tier is at the end
spark.table(CD).printSchema()

In [0]:
# One OLD row (customer_tier is NULL) and one NEW row
display(spark.table(CD).select("customer_id", "first_name", "customer_tier")
        .filter(F.col("customer_id").isin("CUST000001", "CUST010001")))

### Data Quality with Constraints

Use a constraint when a rule must hold for every row, whatever job writes to the table. With a `CHECK` constraint, Delta rejects every write that has a row breaking the rule. `NOT NULL` on a column works the same way. `ADD CONSTRAINT` first checks the rows already in the table and fails if any row breaks the rule. So the next cell removes the rows without `customer_id` before the constraint is added.

In [0]:
# Clean the existing rows first (DELETE is covered below)
display(spark.sql(f"DELETE FROM {CD} WHERE customer_id IS NULL"))

In [0]:
# CHECK constraint: customer_id must start with CUST
spark.sql(f"ALTER TABLE {CD} ADD CONSTRAINT valid_customer_id CHECK (customer_id LIKE 'CUST%')")
print(f"CHECK constraint valid_customer_id added to {CD}")

In [0]:
# A row that breaks the constraint is rejected, and the whole INSERT changes nothing
rows_before = spark.table(CD).count()
try:
    spark.sql(f"""
        INSERT INTO {CD} (customer_id, first_name, last_name, country)
        VALUES ('INVALID123', 'Bad', 'Customer', 'Poland')
    """)
except PySparkException as e:
    msg = str(e)
    lines = msg[max(msg.find("["), 0):].splitlines()  # skip the Java exception name serverless may add
    print(lines[0])
    # The rejected row values follow the first line as " - column : value" lines
    print("\n".join(l for l in lines[1:6] if l.strip().startswith("-")))
else:
    print("Not rejected: the CHECK constraint did not fire")
print(f"Rows before the INSERT: {rows_before}, after: {spark.table(CD).count()}")

In [0]:
# This row passes: 'CUST123' starts with CUST
display(spark.sql(f"""
    INSERT INTO {CD} (customer_id, first_name, last_name, country)
    VALUES ('CUST123', 'Good', 'Customer', 'Poland')
"""))

## CRUD Operations & MERGE

Delta tables support `INSERT`, `UPDATE`, `DELETE` and `MERGE INTO`. Each statement is one atomic commit in the transaction log. It either fully happens or not at all, and it shows up in the table history. In SQL, `display()` of a DML statement shows the number of affected rows.

### INSERT

Use `INSERT` to add new rows to a table.

In [0]:
# INSERT two new customers
display(spark.sql(f"""
    INSERT INTO {CD}
        (customer_id, first_name, last_name, email, phone, city, state, country,
         registration_date, customer_segment, customer_tier)
    VALUES
        ('CUST020001', 'Insert', 'Customer1', 'insert1@example.com', '+48 111 111 111',
         'Warsaw', 'MZ', 'Poland', '2023-12-10', 'Premium', 'Gold'),
        ('CUST020002', 'Insert', 'Customer2', 'insert2@example.com', '+48 222 222 222',
         'Gdansk', 'PM', 'Poland', '2023-12-11', 'Basic', 'Silver')
"""))

### UPDATE

Use `UPDATE` to change column values in the rows that match the `WHERE` condition.

In [0]:
# UPDATE the tier of two customers: num_affected_rows = 2
display(spark.sql(f"""
    UPDATE {CD} SET customer_tier = 'Platinum'
    WHERE customer_id IN ('CUST010001', 'CUST020001')
"""))

### DELETE

Use `DELETE` to remove the rows that match the `WHERE` condition.

In [0]:
# DELETE one customer: num_affected_rows = 1
display(spark.sql(f"DELETE FROM {CD} WHERE customer_id = 'CUST020002'"))

### MERGE INTO (Upsert)

`MERGE` updates the rows that match and inserts the rows that do not, in one atomic commit. Use it when source rows can change, for example CDC data or corrections. For append-only data, a plain `INSERT` is cheaper.

```sql
MERGE INTO target
USING source ON target.key = source.key
WHEN MATCHED THEN UPDATE SET *        -- or SET col = source.col, ...
WHEN NOT MATCHED THEN INSERT *        -- or INSERT (cols) VALUES (...)
WHEN NOT MATCHED BY SOURCE THEN DELETE  -- optional
```

| Clause | Description |
|---|---|
| `WHEN MATCHED` | The source row matches a target row: UPDATE or DELETE |
| `WHEN NOT MATCHED` | The source row has no match in the target: INSERT |
| `WHEN NOT MATCHED BY SOURCE` | The target row has no match in the source: DELETE or UPDATE |
| `AND condition` | Optional extra filter on any `WHEN` clause |

`UPDATE SET *` and `INSERT *` take all columns from the source by name. When the source has fewer or other columns, list the columns explicitly. If **several source rows** match the same target row, MERGE fails. Deduplicate the source first (see *Self-study* at the end).

<img src="../../../assets/images/ff01677d3d4a45d6a6a7530d8911b785.png" width="800">

In [0]:
# MERGE source: one update (CUST010001) and two new customers (same columns as above)
merge_data = spark.createDataFrame([
    ("CUST010001", "Updated", "Name", "updated@example.com", "+48 999 999 999",
     "Poznan", "WP", "Poland", "2023-12-01", "VIP", "Diamond"),
    ("CUST030001", "Brand", "New", "brand.new@example.com", "+48 777 777 777",
     "Wroclaw", "DS", "Poland", "2023-12-15", "Basic", "Bronze"),
    ("CUST030002", "Another", "New", "another.new@example.com", "+48 888 888 888",
     "Lodz", "LD", "Poland", "2023-12-16", "Premium", "Silver"),
], cols).withColumn("registration_date", F.to_date("registration_date"))

# The temp view is the MERGE source
merge_data.createOrReplaceTempView("customer_updates")

In [0]:
# MERGE: CUST010001 is updated, CUST030001 and CUST030002 are inserted
display(spark.sql(f"""
    MERGE INTO {CD} AS target
    USING customer_updates AS source
    ON target.customer_id = source.customer_id
    WHEN MATCHED THEN UPDATE SET *
    WHEN NOT MATCHED THEN INSERT *
"""))

In [0]:
# Check the MERGE: CUST010001 updated, CUST030001 and CUST030002 inserted
ids = ["CUST010001", "CUST030001", "CUST030002"]
display(spark.table(CD).filter(F.col("customer_id").isin(ids)).orderBy("customer_id"))

## Metadata and History

Use these commands to audit who changed what and when, and to find the version you need for time travel.

| Command | Description |
|---|---|
| `DESCRIBE DETAIL table` | One row: format, location, `numFiles`, `sizeInBytes`, partitioning / clustering, properties |
| `DESCRIBE HISTORY table [LIMIT n]` | One row per version: operation, user, timestamp, parameters, metrics |
| `SHOW TBLPROPERTIES table` | Table properties (e.g. retention, constraints) |
| `DESCRIBE EXTENDED table` | Schema, storage info and properties |

### DESCRIBE DETAIL

Use `DESCRIBE DETAIL` when you need to know where a table is stored and how many files it has.

In [0]:
display(spark.sql(f"DESCRIBE DETAIL {CD}").select("format", "location", "numFiles", "sizeInBytes"))

### DESCRIBE HISTORY

Use `DESCRIBE HISTORY` when a table changed and you need to know which operation changed it and when.

In [0]:
# One row per commit. operationMetrics differ per operation
# (MERGE: numTargetRowsInserted / numTargetRowsUpdated, DELETE: numDeletedRows, ...)
display(spark.sql(f"DESCRIBE HISTORY {CD}")
        .select("version", "timestamp", "operation", "operationParameters", "operationMetrics"))

> The history can show `OPTIMIZE` rows you never ran (`auto -> true` in `operationParameters`): on serverless and Unity Catalog managed tables, Databricks runs auto compaction after writes. These commits take version numbers too, so read the version from `DESCRIBE HISTORY` instead of counting your statements, as the time travel demo below does.

## Time Travel and Disaster Recovery

Time travel helps when data was changed or deleted by mistake, or when you need to see a table as it was at an earlier time. Every change creates a new table version. The files of old versions stay in storage until `VACUUM` removes them, so you can read and restore old versions.

| Command | Description |
|---|---|
| `SELECT * FROM t VERSION AS OF n` | Read the table at version `n` |
| `SELECT * FROM t TIMESTAMP AS OF '2026-11-25 10:00:00'` | Read the table as it was at that time |
| `RESTORE TABLE t TO VERSION AS OF n` | Make version `n` the current state (adds a new RESTORE version) |
| `VACUUM t [RETAIN x HOURS]` | Delete unreferenced data files older than the retention (default 7 days). On serverless `RETAIN` is ignored unless it is `0 HOURS` |

<img src="../../../assets/images/nb03-time-travel-v2.png" alt="Time travel on the version timeline with VERSION AS OF, TIMESTAMP AS OF and DESCRIBE HISTORY, and VACUUM moving old, unreferenced files older than 7 days into the bin" width="800">

### Time Travel Setup

A small table with a few known changes makes the versions easy to follow. Each cell below saves the version number it created.

In [0]:
# Create the time travel table. Auto compaction can add versions in between (see the note above),
# so each version number is read from DESCRIBE HISTORY.
# DROP first, so a rerun does not keep the 0-hour retention set later
TT = f"{CATALOG}.{BRONZE_SCHEMA}.time_travel_demo"

def latest_version(table):
    """Return the newest version number from the Delta transaction log."""
    return spark.sql(f"DESCRIBE HISTORY {table} LIMIT 1").first()["version"]

spark.sql(f"DROP TABLE IF EXISTS {TT}")
spark.sql(f"CREATE TABLE {TT} (id INT, name STRING, status STRING, updated_at TIMESTAMP)")
v_create = latest_version(TT)  # CREATE TABLE is a commit too

In [0]:
# Initial data: Alice, Bob, Charlie
spark.sql(f"""
    INSERT INTO {TT} VALUES
        (1, 'Alice', 'active', current_timestamp()),
        (2, 'Bob', 'active', current_timestamp()),
        (3, 'Charlie', 'active', current_timestamp())
""")
v_initial = latest_version(TT)

In [0]:
# Alice is upgraded to premium
spark.sql(f"""
    UPDATE {TT} SET status = 'premium', updated_at = current_timestamp()
    WHERE name = 'Alice'
""")
v_update = latest_version(TT)

In [0]:
# Diana is added
spark.sql(f"INSERT INTO {TT} VALUES (4, 'Diana', 'new', current_timestamp())")
v_insert = latest_version(TT)

In [0]:
# Charlie is deleted
spark.sql(f"DELETE FROM {TT} WHERE name = 'Charlie'")
v_delete = latest_version(TT)
display(spark.table(TT))

### Table History

Compare the history with the table above.

In [0]:
display(spark.sql(f"DESCRIBE HISTORY {TT}")
        .select("version", "timestamp", "operation", "operationParameters"))

### Time Travel Queries

Use `VERSION AS OF` when you know the version number from the history. Use `TIMESTAMP AS OF` when you only know the time when the data was still correct.

In [0]:
# The version with the initial data (Charlie is still there, Alice is still 'active')
display(spark.sql(f"SELECT * FROM {TT} VERSION AS OF {v_initial}"))

In [0]:
# TIMESTAMP AS OF: the table as it was at the commit time of the UPDATE
# (Alice is 'premium', Diana is not added yet)
history = spark.sql(f"DESCRIBE HISTORY {TT}")
ts_update = history.filter(f"version = {v_update}").first()["timestamp"]
print("Timestamp of the UPDATE:", ts_update)
display(spark.sql(f"SELECT * FROM {TT} TIMESTAMP AS OF '{ts_update}'"))

### Disaster Recovery: Accidental Deletion

A `DELETE` without a `WHERE` removes every row in one commit. The cells below simulate this accident, find the last good version in the history, and bring it back with `RESTORE`.

In [0]:
# Simulate the accident: a DELETE without WHERE removes all rows
spark.sql(f"DELETE FROM {TT}")
print("Rows after the accident:", spark.table(TT).count())

In [0]:
display(spark.sql(f"DESCRIBE HISTORY {TT}").select("version", "timestamp", "operation"))

In [0]:
# The accident is the MOST RECENT DELETE (the earlier "Charlie deleted" was a legitimate one)
history = spark.sql(f"DESCRIBE HISTORY {TT}")
accident_version = history.filter("operation = 'DELETE'").agg(F.max("version")).first()[0]
last_good_version = accident_version - 1
print(f"Accidental DELETE = version {accident_version} -> last good version = {last_good_version}")

In [0]:
display(spark.sql(f"RESTORE TABLE {TT} TO VERSION AS OF {last_good_version}"))

In [0]:
# Verify: the rows are back
display(spark.table(TT))

### VACUUM and Its Impact on Time Travel

VACUUM frees storage, but it deletes the files that old versions need. Two separate things limit time travel:
- The table property `delta.deletedFileRetentionDuration` (default 7 days) is a retention guard. Delta rejects time travel to versions older than this value with `DELTA_UNSUPPORTED_TIME_TRAVEL_BEYOND_DELETED_FILE_RETENTION_DURATION`. The check uses the property only, so it fires as soon as you lower it, before any VACUUM.
- VACUUM deletes data files that the current version no longer references and that are older than the retention. It never changes the files of the current version. After VACUUM, those old versions cannot come back.

The demo runs `OPTIMIZE`, lowers the retention to 0 hours and shows that time travel is rejected. Then it runs `VACUUM ... DRY RUN`, the real `VACUUM` and a check. Module 06 shows VACUUM on a bigger table.

> OPTIMIZE runs first because on new tables UPDATE and DELETE write *deletion vectors* instead of rewriting data files, and RESTORE re-adds old files. So old versions still share files with the current one. OPTIMIZE rewrites the small files. After that, the files of all earlier versions are unreferenced and VACUUM can delete them.

> On serverless, the Spark conf `spark.databricks.delta.retentionDurationCheck.enabled` (classic compute) raises `CONFIG_NOT_AVAILABLE`, so the demo lowers the table property instead. On serverless, VACUUM also ignores `RETAIN` unless it is `0 HOURS`.

<img src="../../../assets/images/nb03-time-travel-limits.png" alt="Version timeline with two separate limits: the file retention property that blocks time travel, and VACUUM that deletes old files" width="800">

In [0]:
# A few more rows, so OPTIMIZE has several small files to compact
display(spark.sql(f"""
    INSERT INTO {TT} VALUES
        (5, 'Eve', 'active', current_timestamp()),
        (6, 'Frank', 'active', current_timestamp())
"""))

In [0]:
# OPTIMIZE rewrites the small files: numFilesRemoved / numFilesAdded show the compaction
display(spark.sql(f"OPTIMIZE {TT}").select("metrics.numFilesAdded", "metrics.numFilesRemoved"))

In [0]:
# Files of the current version, right before VACUUM
files_before_vacuum = spark.sql(f"DESCRIBE DETAIL {TT}").first()["numFiles"]
print("numFiles before VACUUM:", files_before_vacuum)

In [0]:
# Lower the file retention to 0 hours (DEMO ONLY: in production keep 7 days or more)
spark.sql(f"""
    ALTER TABLE {TT}
    SET TBLPROPERTIES ('delta.deletedFileRetentionDuration' = 'interval 0 hours')
""")
print(f"{TT}: deletedFileRetentionDuration = 0 hours (demo only)")

In [0]:
# Retention guard: time travel to the first data version is rejected now, before any VACUUM
from pyspark.errors import PySparkException

try:
    spark.sql(f"SELECT * FROM {TT} VERSION AS OF {v_initial}").count()
except PySparkException as e:
    msg = str(e)
    print(msg[max(msg.find("["), 0):].splitlines()[0])  # skip the Java exception name serverless may add
else:
    print(f"Not rejected: version {v_initial} is still readable")

In [0]:
# DRY RUN: lists the files VACUUM would delete (at most the first 1,000), deletes nothing
display(spark.sql(f"VACUUM {TT} RETAIN 0 HOURS DRY RUN"))

In [0]:
# The real VACUUM: deletes all unreferenced files older than the retention
# (DEMO ONLY: never 0 hours in production)
display(spark.sql(f"VACUUM {TT} RETAIN 0 HOURS"))

In [0]:
# After VACUUM: nothing is left to delete, the current version keeps its files and still works
display(spark.sql(f"VACUUM {TT} RETAIN 0 HOURS DRY RUN"))
files_after_vacuum = spark.sql(f"DESCRIBE DETAIL {TT}").first()["numFiles"]
print(f"numFiles before/after VACUUM: {files_before_vacuum}/{files_after_vacuum}")
display(spark.table(TT))

## Change Data Feed (CDF)

`DESCRIBE HISTORY` shows *which operation* made each version. Change Data Feed shows *which rows* changed. With the table property `delta.enableChangeDataFeed = true`, Delta records every inserted, updated and deleted row, and `table_changes()` reads them. A downstream job can then process only the rows that changed since its last run, not the whole table ([Use change data feed](https://learn.microsoft.com/azure/databricks/tables/features/change-data-feed)).

<img src="../../../assets/images/nb03-change-data-feed.png" alt="Change Data Feed on cdf_demo: INSERT of Alice, Bob and Charlie, UPDATE of Alice to premium and DELETE of Bob; table_changes returns six rows with insert, update_preimage, update_postimage and delete, so the next layer reads only changed rows; CDC brings changes from a source system into Delta, CDF passes changes from a Delta table to the next layer" width="800">

The version numbers in your run can differ: auto compaction can add `OPTIMIZE` commits between the statements.

> CDF is not in the 08/2026 exam guide, but it may appear in older practice tests. Change Data Capture (CDC) is a pattern that captures changes from a source system, such as an operational database, and brings them into Delta. Tools like Lakeflow Connect, Debezium or Fivetran do this. Change Data Feed (CDF) is a Delta feature that records changes made within a Delta table. CDC feeds data into Delta. CDF lets the next layer read only what changed.

In [0]:
# Create a table with CDF on from the start, then insert three rows.
# DROP first, so a rerun starts with a clean change feed. The shallow clone (CLONE section
# below) is dropped before its source: some workspaces refuse to drop a base table that
# a shallow clone references (CANNOT_DROP_BASE_TABLE_REFERENCED_BY_SHALLOW_CLONE).
CDF_T = f"{CATALOG}.{BRONZE_SCHEMA}.cdf_demo"
spark.sql(f"DROP TABLE IF EXISTS {CDF_T}_shallow_clone")
spark.sql(f"DROP TABLE IF EXISTS {CDF_T}")
spark.sql(f"""
    CREATE TABLE {CDF_T} (id INT, name STRING, status STRING)
    TBLPROPERTIES (delta.enableChangeDataFeed = true)
""")
spark.sql(f"INSERT INTO {CDF_T} VALUES (1, 'Alice', 'active'), (2, 'Bob', 'active'), "
          f"(3, 'Charlie', 'active')")
v_cdf_insert = latest_version(CDF_T)

In [0]:
# Two changes: UPDATE Alice, DELETE Bob (each statement is a new version)
spark.sql(f"UPDATE {CDF_T} SET status = 'premium' WHERE name = 'Alice'")
spark.sql(f"DELETE FROM {CDF_T} WHERE name = 'Bob'")
display(spark.sql(f"DESCRIBE HISTORY {CDF_T}").select("version", "operation"))

In [0]:
# Read every row-level change since the INSERT version with table_changes()
changes = spark.sql(f"""
    SELECT id, name, status, _change_type, _commit_version
    FROM table_changes('{CDF_T}', {v_cdf_insert})
    ORDER BY _commit_version, id, _change_type DESC
""")
display(changes)

Each change row keeps the row values plus three metadata columns: `_change_type`, `_commit_version` and `_commit_timestamp`. The `OPTIMIZE` commits from auto compaction add no change rows, because they do not change data.

| `_change_type` | Meaning |
|---|---|
| `insert` | New row |
| `update_preimage` | The row **before** the UPDATE |
| `update_postimage` | The row **after** the UPDATE |
| `delete` | Deleted row |

- To turn CDF on for an existing table, run `ALTER TABLE t SET TBLPROPERTIES (delta.enableChangeDataFeed = true)`. Delta records changes only from that version on.
- New managed tables in recent workspaces have row tracking on by default (`delta.enableRowTracking = true` in `SHOW TBLPROPERTIES`). On these tables `table_changes()` returns changes even without the CDF property (checked on 2026-11-23). Set the property anyway: older and external tables need it, and the docs and exams assume it.
- In Python, read the feed with `spark.read.option("readChangeFeed", "true").option("startingVersion", v).table(t)`. `readStream` works the same way for incremental pipelines.

## Managed vs External Tables

Unity Catalog has two kinds of tables. They differ in who owns the data files, and this decides what `DROP TABLE` does.

| | Managed table (default) | External table |
|---|---|---|
| Who owns the files | Unity Catalog: it picks the location in managed storage and maintains the files | You: the files live at the path you give in `LOCATION` |
| Create | `CREATE TABLE t (...)`, no `LOCATION` | `CREATE TABLE t (...) LOCATION 'abfss://...'`. The path must be inside an external location where you have `CREATE EXTERNAL TABLE` |
| Modify | `ALTER TABLE` (columns, properties, owner), the same for both | same |
| `DROP TABLE` | Removes the table and its data. The files are purged after the recovery period (default 7 days). `UNDROP TABLE` brings the table back within that period | Removes only the table from the catalog. The files stay in storage |
| Maintenance | Predictive optimization runs `OPTIMIZE` and `VACUUM` for you | Your team runs it |
| Use it for | Most tables | Files that other tools read directly from storage, or that must outlive the table |

> `df.write.save('/Volumes/...')` writes a path-based Delta folder, not a table. It has no table-level GRANTs and no lineage, and it does not show up in Catalog Explorer. In Unity Catalog, use managed tables, or external tables on an external location.

You can convert a table from one kind to the other:
- External → managed: run `ALTER TABLE t SET MANAGED`. It works on Delta tables only, on Databricks Runtime 17.3 LTS+ or serverless, and you need `OWNER` or `MANAGE` on the table. The data is copied into managed storage, and the name, history and permissions stay. Within 14 days you can roll back with `ALTER TABLE t UNSET MANAGED`.
- Managed → external: there is no command. Copy the data to an external path (`CREATE TABLE ... LOCATION '...' AS SELECT ...` or `DEEP CLONE`), point the readers to the new table, then drop the old one.

<img src="../../../assets/images/nb03-managed-external.png" alt="Managed and external tables side by side: create, DROP and what happens to the files, plus the conversion with SET MANAGED" width="800">

### Demo: Managed Table

Most tables you create are managed. This one has no `LOCATION`, so Unity Catalog decides where the files go. `DESCRIBE TABLE EXTENDED` shows the table type and the location it chose.

In [ ]:
# Managed table: no LOCATION, Unity Catalog chooses where the files go
MT = f"{CATALOG}.{BRONZE_SCHEMA}.managed_demo"
spark.sql(f"CREATE OR REPLACE TABLE {MT} (id INT, name STRING)")
spark.sql(f"INSERT INTO {MT} VALUES (1, 'managed')")
display(spark.sql(f"DESCRIBE TABLE EXTENDED {MT}").filter("col_name IN ('Type', 'Location')"))

In [ ]:
# DROP removes the managed table from the catalog (its files are purged after the recovery period)
spark.sql(f"DROP TABLE {MT}")
print(f"{MT} exists after DROP: {spark.catalog.tableExists(MT)}")

### Demo (trainer only): External Table, DROP and `SET MANAGED`

An external table needs an external location where you have `CREATE EXTERNAL TABLE`, so the trainer runs these cells. The trainer pastes the URL of the training external location into `EXTERNAL_LOCATION_URL`. While it is empty, each cell prints `[SKIPPED]` and does nothing.

The cells create an external table, drop it and list the files that stay in storage. Then they register the same files again and convert the table to managed with `SET MANAGED`.

In [ ]:
# Demo (trainer only): external location URL (empty = skip),
# e.g. abfss://<container>@<account>.dfs.core.windows.net/
EXTERNAL_LOCATION_URL = ""

ET = f"{CATALOG}.{BRONZE_SCHEMA}.external_demo"
# A folder of its own, outside the folders that hold the catalogs' managed storage
ET_PATH = ""
if EXTERNAL_LOCATION_URL:
    ET_PATH = f"{EXTERNAL_LOCATION_URL.rstrip('/')}/m03_external_demo/{CATALOG}"
    print(f"External table path: {ET_PATH}")
else:
    print("[SKIPPED] EXTERNAL_LOCATION_URL is empty: the external table cells do nothing.")

In [ ]:
# Demo (trainer only): create an external table, then DROP it; the files stay in storage
if ET_PATH:
    spark.sql(f"DROP TABLE IF EXISTS {ET}")
    dbutils.fs.rm(ET_PATH, True)  # files from an earlier run would become part of the new table
    spark.sql(f"CREATE TABLE {ET} (id INT, name STRING) LOCATION '{ET_PATH}'")
    spark.sql(f"INSERT INTO {ET} VALUES (1, 'kept after DROP')")
    spark.sql(f"DROP TABLE {ET}")
    left = [f.name for f in dbutils.fs.ls(ET_PATH)]
    print("Files after DROP TABLE:", left)
else:
    print("[SKIPPED] external table demo")

In [ ]:
# Demo (trainer only): register the files again as an external table,
# then convert it to managed in place
if ET_PATH:
    spark.sql(f"CREATE TABLE {ET} LOCATION '{ET_PATH}'")
    spark.sql(f"ALTER TABLE {ET} SET MANAGED")
    details = spark.sql(f"DESCRIBE TABLE EXTENDED {ET}")
    t_type = details.filter("col_name = 'Type'").first()["data_type"]
    rows = spark.table(ET).count()
    print(f"{ET}: {t_type}, {rows} row(s)")
else:
    print("[SKIPPED] SET MANAGED demo")

In [ ]:
# Demo (trainer only): clean up the table and the old external files
# (SET MANAGED keeps them 14 days for UNSET MANAGED)
if ET_PATH:
    spark.sql(f"DROP TABLE IF EXISTS {ET}")
    dbutils.fs.rm(ET_PATH, True)
    print("external_demo dropped, external files removed")
else:
    print("[SKIPPED] cleanup")

## If time allows: CLONE (Shallow vs Deep)

Skip this section when the module runs late. CLONE is not in the 08/2026 exam guide, but it may appear in older practice tests.

`CLONE` copies a Delta table at a version. A deep clone copies the data files. A shallow clone copies only the metadata and reads the files of the source. Shallow clone of Unity Catalog tables is in Public Preview ([Shallow clone for Unity Catalog tables](https://learn.microsoft.com/azure/databricks/tables/operations/clone-unity-catalog)).

| | Shallow clone | Deep clone |
|---|---|---|
| Data files | References the source files, no copy | Full independent copy |
| Speed and storage | Fast, almost no extra storage | Slower, full duplicate |
| Depends on the source | Yes (see the note below) | No |
| Use it for | Short-lived tests and experiments | Backup, migration, environment promotion |

```sql
CREATE TABLE dev.customers_test SHALLOW CLONE prod.customers;
CREATE TABLE backup.customers_bak DEEP CLONE prod.customers;
CREATE TABLE backup.customers_v5 DEEP CLONE prod.customers VERSION AS OF 5;
```

> In Unity Catalog, `VACUUM` on the source does not break a shallow clone, because UC keeps the files that clones still reference. Dropping the source does break it, once the source files are purged after the `UNDROP` period (7 days by default). The rule "VACUUM breaks a shallow clone" is for Hive metastore and path-based tables. `CREATE OR REPLACE` cannot overwrite a shallow clone in Unity Catalog, so drop the clone first.

In [0]:
# Shallow clone of cdf_demo: metadata only, the clone reads the source's data files
SC = f"{CDF_T}_shallow_clone"
spark.sql(f"DROP TABLE IF EXISTS {SC}")
spark.sql(f"CREATE TABLE {SC} SHALLOW CLONE {CDF_T}")
src_rows, sc_rows = spark.table(CDF_T).count(), spark.table(SC).count()
print(f"Source: {src_rows} rows | Shallow clone: {sc_rows} rows")
# The clone has its own history: operation = CLONE, isShallow = true
display(spark.sql(f"DESCRIBE HISTORY {SC}").select("version", "operation", "operationParameters"))

In [0]:
# Deep clone: all data files are copied, so the clone is independent of the source
DC = f"{CDF_T}_deep_clone"
spark.sql(f"CREATE OR REPLACE TABLE {DC} DEEP CLONE {CDF_T}")
print(f"Deep clone: {spark.table(DC).count()} rows")

In [0]:
# Insert a row into the deep clone only: the source does not change
spark.sql(f"INSERT INTO {DC} VALUES (99, 'Clone-only', 'test')")
src_rows, dc_rows = spark.table(CDF_T).count(), spark.table(DC).count()
print(f"Source: {src_rows} rows | Deep clone: {dc_rows} rows")

In [0]:
# Clean up: the clones first, then their source
for t in (SC, DC, CDF_T):
    spark.sql(f"DROP TABLE IF EXISTS {t}")

## Summary

| Concept | What it gives you |
|---|---|
| ACID transactions | Safe concurrent reads and writes, no half-written tables |
| Transaction log | Every change is a commit in `_delta_log/`, the source of truth |
| Schema enforcement / evolution | Bad writes are rejected; `mergeSchema` adds columns on purpose |
| Constraints | `CHECK` / `NOT NULL` reject bad rows at write time |
| MERGE INTO | Upsert in one atomic commit; deduplicate the source on the key first |
| Time travel | `VERSION AS OF n` / `TIMESTAMP AS OF t` |
| RESTORE | Roll back to an earlier version (adds a new version) |
| Change Data Feed | `delta.enableChangeDataFeed = true` records every changed row; `table_changes(t, v)` reads them |
| VACUUM | Deletes data files no longer referenced by the current version, older than the retention (default 7 days) |
| CLONE | Shallow: metadata only, reads the source files; deep: independent copy of the data |
| Managed vs external | Managed: Unity Catalog owns the files; external: you do. `SET MANAGED` converts external → managed |

Rules to remember:
- Use MERGE when rows can change, and a plain INSERT for append-only data.
- Time travel is **not a backup**. `VACUUM` deletes the files old versions need (after `delta.deletedFileRetentionDuration`, default 7 days), and history expires after `delta.logRetentionDuration` (default 30 days).
- OPTIMIZE, Liquid Clustering and predictive optimization are in module 06.

## Self-study

The trainer does not run this part in class, and no lab needs it. It covers identity and generated columns, where the Delta log lives, and the common Delta errors with one runnable example.

### Identity and Generated Columns

Use an identity column when a table needs a surrogate key and the source has none. Use a generated column when a value always follows from other columns, such as a date from a timestamp.

| Syntax | Description |
|---|---|
| `col BIGINT GENERATED ALWAYS AS IDENTITY` | Surrogate key: unique and increasing, but not contiguous |
| `col BIGINT GENERATED BY DEFAULT AS IDENTITY` | Like `ALWAYS`, but you may insert your own values |
| `col TYPE GENERATED ALWAYS AS (expression)` | Computed column, derived from other columns at write time |

In [0]:
# Table with an identity column (order_sk) and a generated column (order_date)
spark.sql(f"""
CREATE OR REPLACE TABLE {CATALOG}.{BRONZE_SCHEMA}.orders_modern (
    order_sk BIGINT GENERATED ALWAYS AS IDENTITY,
    order_id INT,
    total_amount DOUBLE,
    order_timestamp TIMESTAMP,
    order_date DATE GENERATED ALWAYS AS (CAST(order_timestamp AS DATE))
)""")
print(f"Created {CATALOG}.{BRONZE_SCHEMA}.orders_modern")

The `INSERT` below leaves out `order_sk` and `order_date`. Delta generates a unique `order_sk` and computes `order_date` from `order_timestamp`.

In [0]:
display(spark.sql(f"""
    INSERT INTO {CATALOG}.{BRONZE_SCHEMA}.orders_modern (order_id, total_amount, order_timestamp)
    SELECT CAST(id AS INT), 10.0 * id, current_timestamp()
    FROM range(5)
"""))

In [0]:
# order_sk and order_date were filled in by Delta
display(spark.table(f"{CATALOG}.{BRONZE_SCHEMA}.orders_modern").orderBy("order_id"))

### Where the Delta Log Lives

The transaction log is the `_delta_log/` folder next to the data files. Each commit is a JSON file with `add` (new data file), `remove` (a file that is no longer part of the table, e.g. after DELETE or OPTIMIZE) and `commitInfo` (who, when, which operation). The first commit and schema changes also carry `metaData` (schema, properties) and `protocol`. `DESCRIBE HISTORY` reads `commitInfo`, and `DESCRIBE DETAIL` shows the location.

<img src="../../../assets/images/f0361af0b2ef43c0a1f61ee1202e2df3.png" width="800">

In [0]:
# Location of the table in managed storage, and its current files
display(spark.sql(f"DESCRIBE DETAIL {CD}").select("location", "numFiles", "sizeInBytes"))

### Common Errors & Troubleshooting

These are the Delta errors you are most likely to see when you write pipelines.

| Error | Cause | Fix |
|---|---|---|
| `ConcurrentAppendException` | A concurrent write added files to the data that your UPDATE / DELETE / MERGE read | Retry. Make concurrent writes touch disjoint data (explicit predicates, e.g. `WHERE date = ...`). Use row-level concurrency (deletion vectors, liquid clustering). `Serializable` isolation causes more conflicts, not fewer |
| Schema mismatch when writing to a Delta table | Column names or types differ from the table | Check with `DESCRIBE TABLE`. Use `mergeSchema = true` only when the change is intended |
| `DELTA_MULTIPLE_SOURCE_ROW_MATCHING_TARGET_ROW_IN_MERGE` | Several **source** rows match the same **target** row (one source row matching several target rows is allowed) | Deduplicate the source on the merge key before MERGE. The next cell shows the error |
| `DELTA_UNSUPPORTED_TIME_TRAVEL_BEYOND_DELETED_FILE_RETENTION_DURATION` | The version is older than `delta.deletedFileRetentionDuration` | Keep the file retention (default 7 days) at least as long as your time travel window. `delta.logRetentionDuration` (default 30 days) limits the history |
| `DELTA_MISSING_TRANSACTION_LOG` | `_delta_log/` was deleted by hand | Never delete `_delta_log/`. Use `DROP TABLE` |

In [0]:
# MERGE fails when two SOURCE rows match the same TARGET row (the failed MERGE changes nothing)
MD = f"{CATALOG}.{BRONZE_SCHEMA}.merge_dup_demo"
spark.sql(f"CREATE OR REPLACE TABLE {MD} (id INT, name STRING)")
spark.sql(f"INSERT INTO {MD} VALUES (1, 'Alice')")
try:
    spark.sql(f"""
    MERGE INTO {MD} t
    USING (SELECT * FROM VALUES (1, 'Alice A'), (1, 'Alice B') AS s(id, name)) s
    ON t.id = s.id
    WHEN MATCHED THEN UPDATE SET *
    """)
except PySparkException as e:
    msg = str(e)
    print(" ".join(msg[max(msg.find("["), 0):].splitlines()[:2]))  # skip the Java exception name serverless may add
else:
    print("Not rejected: MERGE with a duplicated source key succeeded")
finally:
    spark.sql(f"DROP TABLE IF EXISTS {MD}")

← <a href="$./02_data_ingestion">02 — Data Reading & File Formats</a> | **<a href="$../../..">Course folder (README)</a>** | <a href="$./04_incremental_ingestion">04 — Incremental Ingestion →</a>

## Self-check (trainer): participants can skip this cell

In [0]:
# Self-check (trainer): repeats the error demos on surviving tables and checks the saved results

# Schema enforcement: the unknown column is still rejected
try:
    bad_data.write.mode("append").saveAsTable(CD)
except PySparkException as e:
    first = str(e).splitlines()[0]
    assert "[DELTA_METADATA_MISMATCH]" in first, first
    assert "loyalty_points" in str(e), str(e)[:500]
else:
    raise AssertionError("the write with an unknown column was not rejected")

# CHECK constraint: the bad row is rejected and the table does not change
rows_before = spark.table(CD).count()
try:
    spark.sql(f"INSERT INTO {CD} (customer_id, first_name, last_name, country) "
              f"VALUES ('INVALID123', 'Bad', 'Customer', 'Poland')")
except PySparkException as e:
    first = str(e).splitlines()[0]
    assert "[DELTA_VIOLATE_CONSTRAINT_WITH_VALUES]" in first, first
    assert "valid_customer_id" in str(e), str(e)[:500]
else:
    raise AssertionError("the CHECK constraint did not fire")
assert spark.table(CD).count() == rows_before, "the failed INSERT changed the table"

# VACUUM: nothing left to delete, the current files are unchanged
left_after_vacuum = spark.sql(f"VACUUM {TT} RETAIN 0 HOURS DRY RUN").count()
assert left_after_vacuum == 0, f"{left_after_vacuum} unreferenced files are still there after VACUUM"
assert spark.sql(f"DESCRIBE DETAIL {TT}").first()["numFiles"] == files_before_vacuum, \
    "VACUUM changed the files of the current version"

# Retention guard: the first data version is still rejected after VACUUM
try:
    spark.sql(f"SELECT * FROM {TT} VERSION AS OF {v_initial}").count()
except PySparkException as e:
    first = str(e).splitlines()[0]
    assert "[DELTA_UNSUPPORTED_TIME_TRAVEL_BEYOND_DELETED_FILE_RETENTION_DURATION]" in first, first
else:
    raise AssertionError(f"version {v_initial} was not rejected by the retention guard")

# MERGE with duplicate source rows, repeated on time_travel_demo (the failed MERGE changes nothing)
try:
    spark.sql(f"""
        MERGE INTO {TT} t
        USING (SELECT * FROM VALUES (1, 'Alice A'), (1, 'Alice B') AS s(id, name)) s
        ON t.id = s.id
        WHEN MATCHED THEN UPDATE SET t.name = s.name
    """)
except PySparkException as e:
    first = str(e).splitlines()[0]
    assert "[DELTA_MULTIPLE_SOURCE_ROW_MATCHING_TARGET_ROW_IN_MERGE]" in first, first
else:
    raise AssertionError("MERGE with a duplicated source key did not fail")

# Managed table is gone after DROP; shallow and deep clone row counts (the source did not change in between)
assert not spark.catalog.tableExists(MT), "managed_demo should be gone after DROP"
assert sc_rows == src_rows, (src_rows, sc_rows)
assert dc_rows == src_rows + 1, (src_rows, dc_rows)

# External table demo (trainer only, runs when EXTERNAL_LOCATION_URL is set)
if ET_PATH:
    assert "_delta_log/" in left, f"expected the Delta files to stay after DROP, found {left}"
    assert t_type == "MANAGED" and rows == 1, f"expected MANAGED, 1 row; got {t_type}, {rows}"

print("Self-check passed")